# Business Logic Validation Tests for Wheelie Data Warehouse

This notebook validates business rules, data formats, and calculation correctness.

**Test Categories:**
- **A. Business Logic Rules**: Positive amounts, logical date ordering, valid ranges
- **B. Data Format Validation**: Email formats, phone patterns, domain values
- **C. Calculation Correctness**: Rental amounts, duration calculations, aggregations

**Note:** Required field completeness tests (NOT NULL) are in `test_data_quality.ipynb`

In [ ]:
# ==============================================================================
# TEST CONFIGURATION & HELPER FUNCTIONS
# ==============================================================================
import logging
from pyspark.sql.functions import col, datediff, when

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger("business_logic_tests")

def get_table(table_name: str):
    """Helper to load table from data warehouse."""
    return spark.table(f"wheelie.gold.{table_name}")

def assert_positive_values(table_name: str, column_name: str, allow_zero: bool = False):
    """Assert that a numeric column has only positive values (optionally allowing zero)."""
    df = get_table(table_name)

    if allow_zero:
        invalid = df.filter((col(column_name).isNotNull()) & (col(column_name) < 0))
        condition = ">= 0"
    else:
        invalid = df.filter((col(column_name).isNotNull()) & (col(column_name) <= 0))
        condition = "> 0"

    invalid_count = invalid.count()

    if invalid_count > 0:
        logger.error(f"Invalid values found in {table_name}.{column_name} (expected {condition}):")
        invalid.select(column_name).show(10)

    assert invalid_count == 0, \
        f"{table_name}.{column_name}: Found {invalid_count} invalid value(s). Expected all values {condition}."

def assert_date_ordering(table_name: str, earlier_col: str, later_col: str, allow_equal: bool = False):
    """Assert that dates are in logical order (earlier_col < later_col)."""
    df = get_table(table_name)

    # Filter out rows where either date is null
    df_with_dates = df.filter(col(earlier_col).isNotNull() & col(later_col).isNotNull())

    if allow_equal:
        invalid = df_with_dates.filter(col(earlier_col) > col(later_col))
        condition = f"{earlier_col} <= {later_col}"
    else:
        invalid = df_with_dates.filter(col(earlier_col) >= col(later_col))
        condition = f"{earlier_col} < {later_col}"

    invalid_count = invalid.count()

    if invalid_count > 0:
        logger.error(f"Date ordering violation in {table_name} (expected {condition}):")
        invalid.select(earlier_col, later_col).show(100)

    assert invalid_count == 0, \
        f"{table_name}: Found {invalid_count} row(s) where {condition} is violated."

def assert_email_format(table_name: str, email_col: str):
    """Assert that email column contains valid email format (or NULL for invalid emails)."""
    df = get_table(table_name)

    # Simple email regex pattern
    email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

    # Only check non-NULL emails (invalid emails are set to NULL during cleaning)
    invalid = df.filter(
        (col(email_col).isNotNull()) &
        (~col(email_col).rlike(email_pattern))
    )

    invalid_count = invalid.count()
    null_count = df.filter(col(email_col).isNull()).count()

    if null_count > 0:
        logger.warning(f"{table_name}.{email_col}: {null_count} email(s) were set to NULL (invalid format in source)")

    if invalid_count > 0:
        logger.error(f"Invalid email format in {table_name}.{email_col}:")
        invalid.select(email_col).show(10, truncate=False)

    assert invalid_count == 0, \
        f"{table_name}.{email_col}: Found {invalid_count} invalid email format(s)."

def assert_valid_domain_values(table_name: str, column_name: str, valid_values: list):
    """Assert that a column only contains values from a predefined list."""
    df = get_table(table_name)

    invalid = df.filter(
        (col(column_name).isNotNull()) &
        (~col(column_name).isin(valid_values))
    )

    invalid_count = invalid.count()

    if invalid_count > 0:
        logger.error(f"Invalid domain values in {table_name}.{column_name}:")
        logger.error(f"Expected values: {valid_values}")
        invalid.select(column_name).distinct().show(10, truncate=False)

    assert invalid_count == 0, \
        f"{table_name}.{column_name}: Found {invalid_count} invalid value(s). Expected values: {valid_values}."

def assert_calculation_correctness(table_name: str, calculated_col: str, calculation_expr: str, tolerance: float = 0.01):
    """Assert that a calculated column matches expected calculation."""
    df = get_table(table_name)

    # Add expected calculation column
    df_with_calc = df.withColumn("_expected", eval(calculation_expr))

    # Check for mismatches (with tolerance for floating point)
    invalid = df_with_calc.filter(
        (col(calculated_col).isNotNull()) &
        (col("_expected").isNotNull()) &
        (abs(col(calculated_col) - col("_expected")) > tolerance)
    )

    invalid_count = invalid.count()

    if invalid_count > 0:
        logger.error(f"Calculation mismatch in {table_name}.{calculated_col}:")
        logger.error(f"Expected: {calculation_expr}")
        invalid.select(calculated_col, "_expected").show(10)

    assert invalid_count == 0, \
        f"{table_name}.{calculated_col}: Found {invalid_count} calculation mismatch(es)."

logger.info("BUSINESS LOGIC TEST FRAMEWORK LOADED")
logger.info("=" * 70)

In [ ]:
# ==============================================================================
# TEST DEFINITIONS: A. BUSINESS LOGIC RULES
# ==============================================================================

def test_rental_rate_positive():
    """Test that rental_rate is always positive in dim_car."""
    assert_positive_values("dim_car", "rental_rate", allow_zero=False)

def test_service_cost_positive():
    """Test that service_cost is positive or zero in fact_service."""
    assert_positive_values("fact_service", "service_cost", allow_zero=True)

def test_payment_amount_positive():
    """Test that payment_amount is always positive in fact_rental."""
    assert_positive_values("fact_rental", "payment_amount", allow_zero=False)

def test_rental_amount_positive():
    """Test that rental_amount is always positive in fact_rental."""
    assert_positive_values("fact_rental", "rental_amount", allow_zero=False)

def test_rental_duration_positive():
    """Test that rental_duration is always positive in fact_rental."""
    assert_positive_values("fact_rental", "rental_duration", allow_zero=False)

def test_customer_birth_date_reasonable():
    """Test that customer birth_date is between 1900 and today-18 years."""
    from datetime import datetime, timedelta

    df = get_table("dim_customer")
    min_birth_date = "1900-01-01"
    max_birth_date = (datetime.now() - timedelta(days=18*365)).strftime("%Y-%m-%d")

    invalid = df.filter(
        (col("birth_date").isNotNull()) &
        ((col("birth_date") < min_birth_date) | (col("birth_date") > max_birth_date))
    )

    invalid_count = invalid.count()

    if invalid_count > 0:
        logger.error(f"Invalid birth dates (expected between {min_birth_date} and {max_birth_date}):")
        invalid.select("customer_id", "birth_date").show(10)

    assert invalid_count == 0, \
        f"dim_customer.birth_date: Found {invalid_count} unreasonable birth date(s)."

logger.info("✅ Business logic rule test definitions loaded")

In [ ]:
# ==============================================================================
# TEST DEFINITIONS: B. DATA FORMAT VALIDATION
# ==============================================================================

def test_customer_email_format():
    """Test that customer_email has valid email format in dim_customer."""
    assert_email_format("dim_customer", "customer_email")

def test_fuel_type_domain():
    """Test that fuel_type only contains valid values in dim_car."""
    valid_fuel_types = ["Petrol", "Diesel", "Electric", "Hybrid"]
    assert_valid_domain_values("dim_car", "fuel_type", valid_fuel_types)

logger.info("✅ Data format validation test definitions loaded")


In [ ]:
# ==============================================================================
# TEST DEFINITIONS: C. CALCULATION CORRECTNESS
# ==============================================================================
# Note: Calculation tests removed because fact_rental no longer contains
# the source date columns (return_date) needed to validate calculations.
# rental_duration and rental_amount are calculated in load_all_facts.ipynb
# and trusted to be correct based on the ETL logic.

logger.info("✅ Calculation correctness test definitions loaded")


In [ ]:
# ==============================================================================
# RUN ALL BUSINESS LOGIC TESTS
# ==============================================================================

logger.info("\n" + "=" * 70)
logger.info("EXECUTING ALL BUSINESS LOGIC TESTS")
logger.info("=" * 70 + "\n")

# Collect all test functions
test_functions = [
    # A. Business Logic Rules
    ("A1: dim_car.rental_rate > 0", test_rental_rate_positive),
    ("A2: fact_service.service_cost >= 0", test_service_cost_positive),
    ("A3: fact_rental.payment_amount > 0", test_payment_amount_positive),
    ("A4: fact_rental.rental_amount > 0", test_rental_amount_positive),
    ("A5: fact_rental.rental_duration > 0", test_rental_duration_positive),
    ("A6: customer birth_date reasonable", test_customer_birth_date_reasonable),

    # B. Data Format Validation
    ("B1: customer_email format", test_customer_email_format),
    ("B2: fuel_type domain", test_fuel_type_domain),
]

passed = 0
failed = 0
failed_tests = []

for test_name, test_func in test_functions:
    try:
        logger.info(f"Running: {test_name} - {test_func.__doc__}")
        test_func()
        passed += 1
        logger.info(f"✅ PASS: {test_name}\n")
    except AssertionError as e:
        failed += 1
        failed_tests.append({
            "test": test_name,
            "description": test_func.__doc__,
            "error": str(e)
        })
        logger.error(f"❌ FAIL: {test_name}")
        logger.error(f"   {str(e)}\n")
    except Exception as e:
        failed += 1
        failed_tests.append({
            "test": test_name,
            "description": test_func.__doc__,
            "error": f"Unexpected error: {str(e)}"
        })
        logger.error(f"❌ ERROR: {test_name}")
        logger.error(f"   Unexpected error: {str(e)}\n")

total = len(test_functions)

logger.info("=" * 70)
logger.info("TEST EXECUTION SUMMARY")
logger.info("=" * 70)
logger.info(f"Total Tests: {total}")
logger.info(f"Passed: {passed} ✅")
logger.info(f"Failed: {failed} ❌")
logger.info(f"Success Rate: {(passed/total*100):.1f}%")

if failed > 0:
    logger.error("\n" + "=" * 70)
    logger.error("FAILED TESTS DETAILS")
    logger.error("=" * 70)
    for test in failed_tests:
        logger.error(f"\n❌ {test['test']}")
        logger.error(f"   Description: {test['description']}")
        logger.error(f"   Error: {test['error']}")
    logger.error("\n" + "=" * 70)
    logger.error(f"⚠️  {failed} TEST(S) FAILED - REVIEW REQUIRED")
    logger.error("=" * 70)

    raise Exception(f"Business logic tests failed: {failed}/{total}")
else:
    logger.info("\n" + "=" * 70)
    logger.info("🎉 ALL BUSINESS LOGIC TESTS PASSED!")
    logger.info("=" * 70)
    logger.info("Coverage:")
    logger.info("  • Business Logic Rules: 6 tests (positive amounts, reasonable values)")
    logger.info("  • Data Format Validation: 2 tests (email format, fuel type domain)")
    logger.info("=" * 70)

